In [3]:
# preprocess import numpy as np
from scipy.interpolate import griddata
from datetime import timedelta, datetime, date
import dateutil.parser
import pandas as pd
from scipy.spatial import distance_matrix
import xarray as xr
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(


In [4]:
## 
country = 'cuba' ## FILL WITH THE COUNTRY NAME e.g. cuba
data_path = '' ## FILL WITH THE PATH of OBSERVATIONAL data
coords_path = '../data/'## FILL WITH THE PATH OF THE FILE WITH THE FINAL COORDINATES
data_path   = f'{data_path}allstat_10m_wind.csv' 
coords_path = f'{coords_path}country_fin_coordinates.nc' 


output_path = f'../data/idw_data/{country}/' ## FILL WITH THE PATH of desidered OUTPUT PATH

output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

### Loading the IDW's final coordinates:

In [5]:
#---------------------------------------------------------------------------------------
# High resolution grid obtained as a downscaling output
# Latitudes & Longitudes are stored in a .nc files with the ones of the other countries.
#---------------------------------------------------------------------------------------

def files_reading(country_name):
    coords_fin_path = coords_path
    file_coords     = xr.open_dataset(coords_fin_path)
    lon_fin = (file_coords[f'longitude_{country_name}'].values)[~np.isnan(file_coords[f'longitude_{country_name}'].values)]
    lat_fin = (file_coords[f'latitude_{country_name}'].values)[~np.isnan(file_coords[f'latitude_{country_name}'].values)]
    
    # I want my reshaped grid starting from the upper left point of the country -> flip the latitude in descending order
    lat_fin = np.sort(lat_fin)[::-1]
    
    return lat_fin, lon_fin

In [6]:
lat_idw_new, lon_idw_new = files_reading(country)

### Loading the stations data:

In [7]:
def open_dataset(filepath):
    var_df = pd.read_csv(filepath)
    
    # Retrieving u and v from speed and direction measurements
    direction_rad = np.deg2rad(var_df['dir'])
    var_df['u10'] = - var_df['wds'] * np.sin(direction_rad)
    var_df['v10'] = - var_df['wds'] * np.cos(direction_rad)
    
    # Conversion needed for the monthly resampling
    var_df['time'] = pd.to_datetime(var_df['time'])
    return var_df

In [8]:
wind_df = open_dataset(data_path)

### Data Manipulation:

In [9]:
#---------------------------------------------------------------------------------------------------
# The measurements are every 1 hour
# The IDW function are applied to a monthly dataframe
#---------------------------------------------------------------------------------------------------

def data_manipulation(group):

    var_hourly_res   = group.resample('60min',on='time').mean()
    #daily resampling - apply .max() instead .mean() if you are interested in the monthly max gust
    var_daily_res    = var_hourly_res.reset_index().resample('D',on='time').mean()
    
    #monthly resampling - - apply .max() instead .mean() if you are interested in the monthly max gust
    var_monthly_res  = var_daily_res.reset_index().resample('M', on='time').mean()
    
    #monthly mean for every station (groups)
    var_months       = var_monthly_res.groupby(by=[var_monthly_res.index.month]).mean()

    #Wind direction in degrees from the North, clockwise turn for Cuba

    return var_months

In [10]:
# Apply the procedure for every station 
wind_df_months = wind_df.groupby('id', group_keys=False).apply(lambda x: data_manipulation(x))

In [11]:
#------------------------------------------------
# Applying IDW to IoT data to obtain a consistent map from punctual observations
#
# Parameters:
# - lat: type = array; latitude of locations of IoT sensors;
# - lon: type = array; longitude of locations of IoT sensors;
# - var: type = array; IoT data;
# - lat_new: type = array; latitude of final grid;
# - lon_new: type = array; longitude of final grid.
#
#------------------------------------------------
def apply_idw(pdf, lat_new, lon_new):

    lat = pdf['latitude'].values
    lon = pdf['longitude'].values
    # Locations of the IoT observations
    iot_grid = np.array(list(zip(lat, lon))) 
    # Grid for IoT observation projection
    grid_new = np.array([[x, y] for x in lat_new for y in lon_new])
    # Distance matrix
    dist = distance_matrix(iot_grid, grid_new)**3
    # Weight matrix
    weights = 1.0 / dist
    # Change diagonal values to 1
    np.fill_diagonal(weights, 1) 
    # Normalizing each column of the weight matrix
    weights = weights/np.nansum(weights, axis=0)
    return weights

def simple_idw(pdf, target, weights, lat_new, lon_new):
    z = pdf[target].values
    # Distance matrix
    # Calculating the results of the IDW method
    zi = np.dot(weights.T, z)
    iot_reshape = zi.reshape(len(lat_new), len(lon_new))
    return iot_reshape

In [12]:
#----------------------------------------------------------------------------------------------------
# This function considers a single month, applying the IDW functions defined at the beginning.
# Then it creates a dataframe with the reshaped variable, assigning the high-res final coordinates
# The file is saved
#----------------------------------------------------------------------------------------------------
def execute_idw_monthly(var_df_months, n_month, saving_path, lat_idw_new, lon_idw_new):
    var_df_month = var_df_months[var_df_months.index == n_month]
    weights     = apply_idw(var_df_month, lat_idw_new, lon_idw_new)

    u10_reshaped, v10_reshaped, wds_reshaped, dir_reshaped = [simple_idw(var_df_month, target, weights, lat_idw_new, lon_idw_new) for target in ['u10','v10','wds','dir']]
    u10_idw,v10_idw,wds_idw,dir_idw = [xr.DataArray(var_reshaped, coords=[("latitude", lat_idw_new),("longitude", lon_idw_new)], name=f'{target}_reshaped'
                                                   ) for var_reshaped, target in zip([u10_reshaped, v10_reshaped, wds_reshaped, dir_reshaped], ['u10','v10','wds','dir'])]
    wind_idw_month_xr = xr.merge([u10_idw,v10_idw,wds_idw,dir_idw]).assign_attrs(units= 'm s-1, [°]', 
                                                                                 description = 'The wind direction is measured in degrees North, considering a clockwise turn')
    wind_idw_month_xr.to_netcdf(saving_path+f'10m_wind_idw_{country}_m{n_month}.nc')
    return wind_idw_month_xr

In [13]:
month = 1
wind_idw_month_xr = execute_idw_monthly(wind_df_months, month, output_path, lat_idw_new, lon_idw_new)